# HYDROAWARE Africa project

## Continental Scale Modelling datasets

### Data Availability

The following datasets are compatible with the DRYP hydrological model. The list includes both ready-to-use datasets and preprocessed datasets used to generate inputs consistent with DRYP requirements. All datasets are sourced from public repositories and are available for free download.

### Data Sources

The global and regional datasets included in this list are not exhaustive. Users may select alternative data sources or generate custom datasets, provided they meet the input specifications of the DRYP model.

### Disclaimer

Parameters derived from these datasets are not calibrated. Users are solely responsible for the appropriate application, validation, and calibration of model inputs.

The list is also available in JSON format

### Model parameters

In [22]:
from turtle import home


datasets = {
   "TERRAIN": {
      "path_dem": "/home/c1755103/AF/dataset/raw/hyd_af_dem_30s.tif", # HYDROSHEDS - Digital elevation model
      "path_Qo": None, # initial channel storage
      "path_fdl": "/home/c1755103/AF/dataset/raw/hyd_af_dir_30s.tif", # HYDROSHEDS - flow direction. D8 encoded
      "path_riv_decay": None, # river flow velocity
      "path_mask": "/home/c1755103/AF/dataset/raw/hyd_af_msk_30s.tif", # HYDROSHEDS - basin mask. model active domain
      "path_riv_len": "/home/c1755103/AF/dataset/postpp/af_river_length_30s.tif", # river length
      "path_riv_width": "/home/c1755103/AF/dataset/postpp/af_river_width_30s.tif", # river width
      "path_riv_elev": "/home/c1755103/AF/dataset/postpp/af_dem_30s_q1.tif", # river stream bottom elevation, quartil 1 of the DEM
      "path_of_bc_flux": None, # flux boundary condition overland flow
      "path_acc": "/home/c1755103/AF/dataset/raw/hyd_af_acc_30s.tif", # HYDROSHEDS - flow accumulation
   },

   "UNSATURATED": {
      "path_uz_theta_sat": "/home/c1755103/AF/dataset/postpp/af_theta_s_l1.tif", # saturated water content
      "path_uz_theta_awc": "/home/c1755103/AF/dataset/postpp/af_theta_available_water_content.asc", # available water content
      "path_uz_theta_wp": "/home/c1755103/AF/dataset/postpp/af_theta_wilting_point.asc", # wilting point
      "path_uz_rootdepth": "/home/c1755103/AF/dataset/postpp/gyga_af_agg_erzd_pwp__m_1km_reprojected.tif", # effective rooting depth
      "path_uz_lambda": "/home/c1755103/AF/dataset/postpp/af_lambda_l1.tif", # lambda parameter
      "path_uz_psi": "/home/c1755103/AF/dataset/postpp/af_psi_s_l1.tif", # soil water potential
      "path_uz_ksat": "/home/c1755103/AF/dataset/postpp/af_k_s_l1.tif", # saturated hydraulic conductivity
      "path_uz_theta": None, # initial conditions
      "path_riv_ksat": "/home/c1755103/AF/dataset/postpp/af_k_s_l1.tif",
      "path_uz_bottomksat": "/home/c1755103/AF/dataset/postpp/af_k_s_l1.tif",
      "path_uz_bc_flux": None,
   },

   "SATURATED": {
      "path_sz_mask": None,
      "path_sz_ksat": "/home/c1755103/AF/dataset/postpp/af_glyhmps_ksat.tif", # GLHYMPS - Global hydrogeology maps
      "path_sz_sy": "/home/c1755103/AF/dataset/postpp/af_glyhmps_sy.tif", # GLHYMPS - Global hydrogeology maps
      "path_sz_wte": "/home/c1755103/AF/dataset/postpp/af_water_table_depth_30s.tif", # water table depth GLOBWB model
      "path_sz_bc_flux": None,
      "path_sz_bc_head": "/home/c1755103/AF/dataset/postpp/af_head_boundary_basin_lvl0.tif",
      "path_sz_bottom": None,
      "path_sz_depth": "/home/c1755103/AF/dataset/postpp/af_average_soil_and_sedimentary-deposit_thickness.tif", # average soil and sedimentary deposit thickness
      "path_sz_bdd": None,
      "path_sz_type": None, # aquifer type
   },
   "INTERCEPTION": {
      "path_veg_hs_extinction_depth": None,
      "path_veg_rp_extinction_depth": None,
   },

   "WATER_BODIES": {
      "path_lake_ids": None, # lakes labels as integers
      "path_lake_depth": "/home/c1755103/AF/dataset/postpp/af_bathymetry_30s.tif", # lakes bathymetry
      "path_pnd_hmax": None, # ponds max depth
      "path_pnd_Amax": None, # ponds maximum extend
      "path_pnd_Vo": None, # ponds volume of water
      "path_wb_bc_flux": None, # water bodies boundary conditions
      "path_slks_depth": None, # shallow lakes depth
      "path_slks_area": None, # shallow lakes area
   }
}

In [23]:
# Save dictionary to a json file
import json

fname = "/home/c1755103/AF/dataset/postpp/af_global_parameters.json"

with open(fname, "w") as f:
    json.dump(datasets, f, indent=4)

### Check that all processed datasets have the same resolution

In [24]:
# check if the saved raster has the same shape and bounds as the reference raster
import rasterio
def check_raster_match(out_path, ref_path):
    with rasterio.open(out_path) as dst:
        out_bounds = dst.bounds
        out_width = dst.width
        out_height = dst.height

    # check if the saved raster has the same shape and bounds as the reference raster
    with rasterio.open(ref_path) as ref:
        ref_bounds = ref.bounds
        ref_width = ref.width
        ref_height = ref.height

    if (out_bounds == ref_bounds) and (out_width == ref_width) and (out_height == ref_height):
        print("Ok")
    else:
        print("The saved raster does not match the reference raster.")

In [25]:
# Check that all datasets have the same resolution and extent
import os
import shutil
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling


def compare_and_resample_to_reference(first_raster, ref_raster, output_raster=None, tol=1e-9):
    """
    Compare extent and resolution between two rasters.
    If they differ, resample the first raster to match the second (reference)
    using nearest-neighbor.

    Parameters
    ----------
    first_raster : str
        Path to the raster to be checked/resampled.
    ref_raster : str
        Path to the reference raster.
    output_raster : str or None
        Output path. If None, saves next to first raster with suffix '_resampled'.
    tol : float
        Tolerance for float comparisons.

    Returns
    -------
    dict
        Summary with match flags and output path.
    """
    with rasterio.open(first_raster) as src, rasterio.open(ref_raster) as ref:
        # Compare extent and pixel size only, as requested.
        same_extent = all(np.isclose(src.bounds[i], ref.bounds[i], atol=tol) for i in range(4))
        same_resolution = np.isclose(src.res[0], ref.res[0], atol=tol) and np.isclose(src.res[1], ref.res[1], atol=tol)

        if output_raster is None:
            base, ext = os.path.splitext(first_raster)
            output_raster = f"{base}_resampled{ext}"

        if same_extent and same_resolution:
            if os.path.abspath(first_raster) != os.path.abspath(output_raster):
                shutil.copy(first_raster, output_raster)
            return {
                "same_extent": True,
                "same_resolution": True,
                "resampled": False,
                "output_raster": output_raster,
            }

        dst_profile = src.profile.copy()
        dst_profile.update({
            "height": ref.height,
            "width": ref.width,
            "transform": ref.transform,
            "crs": ref.crs,
        })

        with rasterio.open(output_raster, "w", **dst_profile) as dst:
            for band_idx in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, band_idx),
                    destination=rasterio.band(dst, band_idx),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref.transform,
                    dst_crs=ref.crs,
                    resampling=Resampling.nearest,
                )

    return {
        "same_extent": same_extent,
        "same_resolution": same_resolution,
        "resampled": True,
        "output_raster": output_raster,
    }


In [26]:
# specify the reference raster path
ref_raster = "/home/c1755103/AF/dataset/raw/hyd_af_dem_30s.tif"  # reference raster

# loop through all datasets and check/resample rasters
for category, paths in datasets.items():
    print(f"Processing category: {category}")
    for key, path in paths.items():
        if path is not None:
            print(f"  Checking {key}...")
            result = compare_and_resample_to_reference(path, ref_raster, output_raster=path)  # overwrite original path
            print(f"    same_extent: {result['same_extent']}, same_resolution: {result['same_resolution']}, resampled: {result['resampled']}, output_raster: {result['output_raster']}")

Processing category: TERRAIN
  Checking path_dem...
    same_extent: True, same_resolution: True, resampled: False, output_raster: /home/c1755103/AF/dataset/raw/hyd_af_dem_30s.tif
  Checking path_fdl...
    same_extent: True, same_resolution: True, resampled: False, output_raster: /home/c1755103/AF/dataset/raw/hyd_af_dir_30s.tif
  Checking path_mask...
    same_extent: True, same_resolution: True, resampled: False, output_raster: /home/c1755103/AF/dataset/raw/hyd_af_msk_30s.tif
  Checking path_riv_len...
    same_extent: False, same_resolution: True, resampled: True, output_raster: /home/c1755103/AF/dataset/postpp/af_river_length_30s.tif
  Checking path_riv_width...
    same_extent: False, same_resolution: True, resampled: True, output_raster: /home/c1755103/AF/dataset/postpp/af_river_width_30s.tif
  Checking path_riv_elev...
    same_extent: True, same_resolution: True, resampled: False, output_raster: /home/c1755103/AF/dataset/postpp/af_dem_30s_q1.tif
  Checking path_acc...
    same_

### Calculation of global parameters for the model setup.
These parameters are used in the model configuration file and are derived from the datasets above.
They include spatial resolution, domain extent, and other relevant parameters for the hydrological model.

In [2]:
import os
import sys
sys.path.append('/home/c1755103/gitremote/CUWALID')
import cuwalid.tools.DRYP_rrtools as rrtools

In [3]:
# calculate wilting point and available water content
rrtools.create_raster_soil_parameters(
    datasets["UNSATURATED"]["path_uz_theta_sat"],
    datasets["UNSATURATED"]["path_uz_psi"],
    datasets["UNSATURATED"]["path_uz_lambda"],
    name_out="af_theta_",)

Raster files saved: 
/home/c1755103/AF/dataset/postpp/af_theta_field_capacity.asc,
/home/c1755103/AF/dataset/postpp/af_theta_wilting_point.asc,
/home/c1755103/AF/dataset/postpp/af_theta_available_water_content.asc


### Calculate river network

### Flatten surface water bodies 